In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import joblib
import onnxruntime as ort
import numpy as np
from src.helpers import cst_to_coords
from pathlib import Path
from aerosandbox import Airfoil
from src.airfoil import airfoil_modifications

# Find the project root (the nearest ancestor that contains `src`) and add it to sys.path
cwd = Path.cwd()
project_root = cwd
for _ in range(6):
    if (project_root / "src").exists():
        break
    if project_root.parent == project_root:
        break
    project_root = project_root.parent
else:
    project_root = cwd

proj_path = str(project_root.resolve())
if proj_path not in sys.path:
    # Insert at front so local packages take precedence
    sys.path.insert(0, proj_path)

Failed to read module file 'C:\Users\matso\AppData\Local\Python\pythoncore-3.14-64\Lib\_strptime.py' for module '_strptime': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\matso\OneDrive\Desktop\Code\Py\AeroRL-Optimizer\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "c:\Users\matso\OneDrive\Desktop\Code\Py\AeroRL-Optimizer\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
  File "C:\Users\matso\AppData\Local\Python\pythoncore-3.14-64\Lib\importlib\__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1398, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1371, in _find_and_load
  File "<frozen importlib.

In [2]:
RUN_ID = "20260221-165429"
LATENT_DIM = 16

BASE_DIR = Path(proj_path)
MODELS_PATH = BASE_DIR / "models"
ONNX_MODEL_PATH = MODELS_PATH / "onnx_decoder"

SCALER_PATH = ONNX_MODEL_PATH / f"{RUN_ID}_scaler.pkl"
DECODER_PATH = ONNX_MODEL_PATH / f"{RUN_ID}_decoder.onnx"

In [97]:
# ============================================================================
# 2. INFERÊNCIA E PLOTAGEM
# ============================================================================
# 1. Carregar o Scaler
scaler = joblib.load(SCALER_PATH)

# 2. Iniciar Sessão ONNX
session = ort.InferenceSession(DECODER_PATH)
input_name = session.get_inputs()[0].name

# 3. Criar Vetor Latente Aleatório (Batch=1, LatentDim=16)
# A distribuição normal garante que o formato gerado seja realista
z_random = np.random.normal(0, 1.0, size=(1, LATENT_DIM)).astype(np.float32)

# 4. Rodar Inferência
outputs = session.run(None, {input_name: z_random})

# O ONNX retorna os tensores crus (w_norm e p_norm)
w_norm = outputs[0] 
p_norm = outputs[1] 

# 5. Desnormalizar os dados
w_phys, p_phys = scaler.inverse_transform(w_norm, p_norm)

# 6. Gerar Coordenadas CST
x_coords, y_coords = cst_to_coords(w_phys[0], p_phys[0], n_points=100)

coords = np.stack((x_coords, y_coords), axis=-1)

af = Airfoil(coordinates=coords)
af.draw()